# Módulo 5: Cálculo II
## Lección 3: Ecuaciones Diferenciales Ordinarias de Primer Orden

Las **Ecuaciones Diferenciales Ordinarias (EDOs)** constituyen el lenguaje fundamental de la física teórica y aplicada. Desde la formulación de las leyes del movimiento de Newton hasta la mecánica cuántica y la relatividad, el modelado del cambio continuo en la naturaleza requiere relacionar cantidades físicas con sus tasas de variación temporal o espacial.

En esta lección, exploraremos de manera formal y rigurosa la teoría de las ecuaciones diferenciales ordinarias de primer orden. Analizaremos los métodos clásicos de resolución analítica, la visualización geométrica mediante campos de direcciones y la aproximación computacional a través de algoritmos numéricos como el método de Euler y Runge-Kutta de cuarto orden (RK4).

### Objetivos de Aprendizaje

Al finalizar esta lección, serás capaz de:
1. **Clasificar EDOs** según su orden, grado, linealidad y homogeneidad.
2. **Resolver analíticamente** tipos clásicos de EDOs de primer orden (Variables separables, homogéneas, lineales, exactas con factores integrantes, Bernoulli y Riccati).
3. **Graficar e interpretar geométricamente** campos de direcciones para comprender el comportamiento cualitativo de las soluciones.
4. **Implementar desde cero** algoritmos numéricos didácticos (Euler y RK4) para la resolución de Problemas de Valor Inicial (PVI).
5. **Validar la precisión y convergencia** de tus implementaciones numéricas mediante la comparación con resolvedores analíticos simbólicos (`SymPy`) e industriales (`SciPy`), analizando cuantitativamente las tasas de convergencia global en gráficos log-log.

### 1. Definiciones y Clasificación de EDOs

Una **Ecuación Diferencial Ordinaria (EDO)** es una relación matemática que vincula una variable independiente $x$, una función desconocida (o variable dependiente) $y(x)$ y sus derivadas ordinarias:

$$F\left(x, y, y', y'', \dots, y^{(n)}\right) = 0$$

#### Clasificación Fundamental:
- **Orden:** Es la derivada de mayor orden presente en la ecuación. Por ejemplo, en $y'' + 3y' - y = \sin(x)$, el orden es $2$.
- **Grado:** Es el exponente algebraico de la derivada de mayor orden cuando la ecuación está expresada en forma polinómica respecto de sus derivadas.
- **Linealidad:** Una EDO de orden $n$ es **lineal** si se puede escribir en la forma:
  
  $$a_n(x) y^{(n)} + a_{n-1}(x) y^{(n-1)} + \dots + a_1(x) y' + a_0(x) y = g(x)$$
  
  donde los coeficientes $a_i(x)$ y el término fuente $g(x)$ dependen únicamente de la variable independiente $x$. Si no es posible escribirla de esta forma (por ejemplo, debido a términos como $y^2$, $e^y$ o $y y'$), la ecuación es **no lineal**.

Una EDO de **primer orden** expresada en forma explícita o normal se denota como:

$$y' = f(x, y)$$

### 2. Interpretación Geométrica: Campo de Direcciones

Geométricamente, una EDO de primer orden $y' = f(x,y)$ define la **pendiente** de la curva solución $y(x)$ en cualquier punto $(x, y)$ del plano real. 

Si dibujamos un pequeño segmento o vector unitario con pendiente $f(x, y)$ en cada punto de una rejilla en el plano, obtenemos un **campo de direcciones** (o *slope field*). Las soluciones de la ecuación diferencial, llamadas **curvas integrales**, son curvas tangentes a estos segmentos en cada punto de su trayectoria.

A continuación, implementaremos una función didáctica en Python utilizando NumPy y Matplotlib para generar y graficar el campo de direcciones de una EDO general.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp

# Configuración del estilo gráfico para Matplotlib
plt.style.use('seaborn-v0_8-whitegrid')

def graficar_campo_direcciones(f, x_lim, y_lim, grid_res=20, ax=None):
    """
    Dibuja el campo de direcciones (vectores unitarios) para la EDO dy/dx = f(x, y).
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
        
    x_vals = np.linspace(x_lim[0], x_lim[1], grid_res)
    y_vals = np.linspace(y_lim[0], y_lim[1], grid_res)
    X, Y = np.meshgrid(x_vals, y_vals)
    
    # Calcular las componentes horizontal (DX) y vertical (DY)
    DX = np.ones_like(X)
    DY = f(X, Y)
    
    # Normalizar los vectores para que tengan longitud unitaria y no se solapen
    magnitud = np.hypot(DX, DY)
    DX /= magnitud
    DY /= magnitud
    
    ax.quiver(X, Y, DX, DY, color='gray', alpha=0.5, headwidth=3, headlength=4, label='Campo de Direcciones')
    ax.set_xlabel('Variable independiente $x$')
    ax.set_ylabel('Variable dependiente $y$')
    ax.set_xlim(x_lim)
    ax.set_ylim(y_lim)
    ax.grid(True, linestyle='--', alpha=0.5)

### 3. Métodos Analíticos de Resolución de EDOs de Primer Orden

Analizaremos los seis métodos clásicos y fundamentales para resolver de manera exacta EDOs de primer orden.

#### A. Variables Separables
Una ecuación es separable si se puede factorizar en la forma:

$$\frac{dy}{dx} = g(x)h(y)$$

Para resolverla, separamos las variables e integramos ambos lados directamente:

$$\int \frac{1}{h(y)} dy = \int g(x) dx + C$$

#### B. Ecuaciones Homogéneas
Una ecuación es homogénea de primer orden si se puede expresar como:

$$\frac{dy}{dx} = F\left(\frac{y}{x}\right)$$

Se resuelve mediante el cambio de variable $v = \frac{y}{x} \implies y = vx \implies \frac{dy}{dx} = v + x\frac{dv}{dx}$. Sustituyendo esto, se reduce a una ecuación separable para $v(x)$:

$$v + x\frac{dv}{dx} = F(v) \implies \frac{1}{F(v) - v} dv = \frac{1}{x} dx$$

#### C. Ecuaciones Lineales de Primer Orden
Tienen la forma general:

$$\frac{dy}{dx} + P(x)y = Q(x)$$

Multiplicando toda la ecuación por el **factor integrante** $\mu(x) = e^{\int P(x) dx}$, el lado izquierdo se convierte en la derivada de un producto:

$$\frac{d}{dx}\left[ e^{\int P(x) dx} y \right] = Q(x) e^{\int P(x) dx}$$

Integrando a ambos lados, obtenemos la solución explícita:

$$y(x) = e^{-\int P(x) dx} \left[ \int Q(x) e^{\int P(x) dx} dx + C \right]$$

#### D. Ecuaciones Exactas y Factores Integrantes
Una ecuación de la forma:

$$M(x, y) dx + N(x, y) dy = 0$$

es **exacta** si existe una función potencial $\Psi(x, y)$ tal que $d\Psi = M dx + N dy$. La condición necesaria y suficiente para que sea exacta es:

$$\frac{\partial M}{\partial y} = \frac{\partial N}{\partial x}$$

La solución general está dada de forma implícita por $\Psi(x, y) = C$.
Si la ecuación no es exacta, a veces puede convertirse en exacta multiplicándola por un **factor integrante** $\mu(x, y)$ que cumpla con la ecuación de exactitud.

#### E. Ecuación de Bernoulli
Tiene la forma no lineal:

$$\frac{dy}{dx} + P(x)y = Q(x)y^n, \quad (n \neq 0, 1)$$

Se linealiza mediante la sustitución $v = y^{1-n} \implies \frac{dv}{dx} = (1-n)y^{-n}\frac{dy}{dx}$. Multiplicando la EDO original por $(1-n)y^{-n}$ se obtiene una ecuación lineal para $v(x)$:

$$\frac{dv}{dx} + (1-n)P(x)v = (1-n)Q(x)$$

#### F. Ecuación de Riccati
Tiene la forma no lineal de segundo grado:

$$\frac{dy}{dx} + P(x)y + Q(x)y^2 = R(x)$$

Si conocemos una solución particular $y_1(x)$, podemos realizar la sustitución $y(x) = y_1(x) + \frac{1}{u(x)} \implies y' = y_1' - \frac{u'}{u^2}$. Esto reduce la EDO de Riccati a una EDO lineal de primer orden en la variable $u(x)$.

### 4. Resolución Simbólica con SymPy

Para ilustrar de forma práctica la potencia matemática de Python, utilizaremos la biblioteca `SymPy` para obtener la solución general exacta y simbólica de un ejemplo clásico de EDO lineal de primer orden:

$$\frac{dy}{dx} - 2xy = x$$

In [ ]:
# Definición de variables simbólicas
x = sp.Symbol('x')
y = sp.Function('y')

# Enunciar la ecuación diferencial simbólicamente: y'(x) - 2*x*y(x) - x = 0
edo = sp.Eq(y(x).diff(x) - 2*x*y(x), x)

# Resolver la EDO
solucion_simbolica = sp.dsolve(edo, y(x))

print("Ecuación Diferencial a resolver:")
sp.pprint(edo)
print("\nSolución general exacta obtenida:")
sp.pprint(solucion_simbolica)

### 5. Métodos Numéricos para Problemas de Valor Inicial (PVI)

En muchas aplicaciones científicas y físicas, las EDOs resultantes son altamente no lineales y no poseen soluciones analíticas en términos de funciones elementales. En estos casos, debemos recurrir a **métodos numéricos** de integración.

Consideramos el **Problema de Valor Inicial (PVI)** o de Cauchy:

$$\frac{dy}{dx} = f(x, y), \quad y(x_0) = y_0$$

Discretizamos el dominio en pasos finitos de tamaño $h$: $x_n = x_0 + n \cdot h$.

#### A. Método de Euler (Primer Orden)
El método de Euler es el esquema numérico más simple. Se obtiene truncando la serie de Taylor de $y(x_{n+1})$ en el término de primer orden:

$$y(x_{n+1}) \approx y(x_n) + h \cdot y'(x_n)$$

Dado que $y'(x_n) = f(x_n, y_n)$, la relación de recurrencia es:

$$y_{n+1} = y_n + h \cdot f(x_n, y_n)$$

Este método tiene un **error de truncamiento local** de $O(h^2)$ y un **error global acumulado** de $O(h)$, por lo que se clasifica como un método de **primer orden**.

#### B. Método de Runge-Kutta de Cuarto Orden (RK4)
El método RK4 proporciona una aproximación de mucho mayor orden y estabilidad evaluando la pendiente en múltiples puntos intermedios del intervalo $[x_n, x_n + h]$. Su fórmula de recurrencia es:

$$y_{n+1} = y_n + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

donde los coeficientes intermedios son:
- $k_1 = f(x_n, y_n)$ (pendiente al inicio del intervalo)
- $k_2 = f\left(x_n + \frac{h}{2}, y_n + h \frac{k_1}{2}\right)$ (pendiente aproximada en el punto medio)
- $k_3 = f\left(x_n + \frac{h}{2}, y_n + h \frac{k_2}{2}\right)$ (pendiente corregida en el punto medio)
- $k_4 = f(x_n + h, y_n + h k_3)$ (pendiente estimada al final del intervalo)

Este esquema posee un **error global de truncamiento** de $O(h^4)$, lo que significa que si reducimos el paso $h$ a la mitad, el error global disminuye por un factor de $2^4 = 16$.

Implementemos estos resolvedores didácticos.

In [ ]:
def resolver_euler(f, x0, y0, xf, h):
    """
    Resuelve el PVI dy/dx = f(x, y) usando el método de Euler.
    """
    x_valores = np.arange(x0, xf + h/2, h)
    y_valores = np.zeros(len(x_valores))
    y_valores[0] = y0
    
    for i in range(len(x_valores) - 1):
        y_valores[i+1] = y_valores[i] + h * f(x_valores[i], y_valores[i])
        
    return x_valores, y_valores

def resolver_rk4(f, x0, y0, xf, h):
    """
    Resuelve el PVI dy/dx = f(x, y) usando el método de Runge-Kutta de 4to Orden (RK4).
    """
    x_valores = np.arange(x0, xf + h/2, h)
    y_valores = np.zeros(len(x_valores))
    y_valores[0] = y0
    
    for i in range(len(x_valores) - 1):
        x_n = x_valores[i]
        y_n = y_valores[i]
        
        k1 = f(x_n, y_n)
        k2 = f(x_n + h/2, y_n + h*k1/2)
        k3 = f(x_n + h/2, y_n + h*k2/2)
        k4 = f(x_n + h, y_n + h*k3)
        
        y_valores[i+1] = y_n + (h / 6.0) * (k1 + 2*k2 + 2*k3 + k4)
        
    return x_valores, y_valores

### 6. Estudio de Caso Físico: Caída Libre con Resistencia Cuadrática del Aire

Consideremos un objeto de masa $m = 80\text{ kg}$ (por ejemplo, un paracaidista en caída libre) que experimenta la fuerza de gravedad y la resistencia viscosa del aire proporcional al cuadrado de su velocidad. De acuerdo con la segunda ley de Newton:

$$m \frac{dv}{dt} = mg - k v^2$$

Dividiendo entre la masa $m$, obtenemos la ecuación diferencial de la velocidad:

$$\frac{dv}{dt} = g - \frac{k}{m} v^2$$

#### Análisis de la EDO:
1. **Clasificación:** Es una EDO de primer orden no lineal. Es autónoma, separable y también es una ecuación del tipo **Riccati** ($v' + Q(t)v^2 = R(t)$).
2. **Velocidad Terminal:** Se alcanza cuando la aceleración es nula ($dv/dt = 0$):
   
   $$g - \frac{k}{m} v_{\infty}^2 = 0 \implies v_{\infty} = \sqrt{\frac{m g}{k}}$$

3. **Solución Analítica Exacta:** Separando variables e integrando con la condición inicial $v(0) = 0$, obtenemos la velocidad en función del tiempo:
   
   $$v(t) = v_{\infty} \tanh\left(\frac{g t}{v_{\infty}}\right)$$

para $v(0) = 0$.

Procederemos a simular este sistema numéricamente utilizando nuestros resolutores artesanales y a graficar las trayectorias resultantes junto al campo de direcciones.

In [ ]:
# Parámetros físicos del sistema
g_acel = 9.81      # Aceleración de gravedad (m/s^2)
m_masa = 80.0      # Masa del paracaidista (kg)
k_drag = 0.25      # Coeficiente aerodinámico (kg/m)

# Calcular velocidad terminal teórica
v_term = np.sqrt((m_masa * g_acel) / k_drag)
print(f"Velocidad terminal teórica: {v_term:.4f} m/s (~{v_term*3.6:.1f} km/h)")

# Definición de la función f(t, v) de la EDO dv/dt = f(t, v)
def dv_dt(t, v):
    return g_acel - (k_drag / m_masa) * v**2

# Solución analítica exacta
def velocidad_exacta(t):
    return v_term * np.tanh((g_acel * t) / v_term)

# Parámetros de integración
t0, tf = 0.0, 15.0
v0 = 0.0
h_paso = 0.5

# Simular con Euler y RK4 artesanales
t_e, v_e = resolver_euler(dv_dt, t0, v0, tf, h_paso)
t_r, v_r = resolver_rk4(dv_dt, t0, v0, tf, h_paso)

# Graficar
fig, ax = plt.subplots(figsize=(10, 6))

# Dibujar el campo de direcciones del plano (t, v)
graficar_campo_direcciones(dv_dt, [t0, tf], [0, v_term * 1.2], grid_res=20, ax=ax)

# Curvas de solución
t_exacto = np.linspace(t0, tf, 200)
ax.plot(t_exacto, velocidad_exacta(t_exacto), 'k-', linewidth=2.5, label='Solución Analítica Exacta')
ax.plot(t_e, v_e, 'ro--', alpha=0.8, label=f'Euler Manual (h={h_paso})')
ax.plot(t_r, v_r, 'bs-.', alpha=0.8, label=f'RK4 Manual (h={h_paso})')

# Línea de la velocidad terminal de referencia
ax.axhline(v_term, color='green', linestyle=':', linewidth=2, label=f'Velocidad Terminal ({v_term:.2f} m/s)')

ax.set_title('Campo de Direcciones y Curvas de Velocidad para el Paracaidista', fontsize=12, fontweight='bold')
ax.set_xlabel('Tiempo $t$ (segundos)')
ax.set_ylabel('Velocidad $v$ (m/s)')
ax.legend(loc='lower right', frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

### 7. Capa de Verificación y Análisis de Convergencia

Para validar científicamente nuestro resolvedor, utilizaremos un doble método de verificación cuantitativa:

1. **Comparación contra SciPy:** Usaremos `scipy.integrate.solve_ivp` con un resolvedor adaptativo RK45 de alta precisión como estándar de referencia de software libre.
2. **Análisis de la Pendiente del Error (Convergencia Global):**
   Calculamos la velocidad aproximada en un tiempo final $t_{fin} = 10.0\text{ s}$ utilizando diferentes tamaños de paso $h$:
   
   $$h \in \{1.0, 0.5, 0.25, 0.1, 0.05, 0.02, 0.01, 0.005\}$$
   
   Evaluamos el error absoluto con respecto a la solución analítica exacta:
   
   $$\text{Error}(h) = |v_{aproximado}(t_{fin}; h) - v_{exacto}(t_{fin})|$$
   
   Al graficar $\text{Error}(h)$ vs. $h$ en escala log-log, la relación teórica debe ser lineal:
   
   $$\log(\text{Error}(h)) \approx m \log(h) + b$$
   
   donde la pendiente $m$ representa el **orden de convergencia global** del método ($m \approx 1$ para Euler y $m \approx 4$ para RK4).

In [ ]:
# Definición de pasos temporales
lista_h = [1.0, 0.5, 0.25, 0.1, 0.05, 0.02, 0.01, 0.005]
errores_e = []
errores_r = []

t_fin = 10.0
val_exacto_fin = velocidad_exacta(t_fin)

for h in lista_h:
    # Simular
    _, v_e_res = resolver_euler(dv_dt, t0, v0, t_fin, h)
    _, v_r_res = resolver_rk4(dv_dt, t0, v0, t_fin, h)
    
    # Tomar el último valor correspondiente a t_fin
    errores_e.append(abs(v_e_res[-1] - val_exacto_fin))
    errores_r.append(abs(v_r_res[-1] - val_exacto_fin))

# Ajustes lineales en escala logarítmica para estimar las pendientes
log_h = np.log(lista_h)
pend_e = np.polyfit(log_h, np.log(errores_e), 1)[0]
pend_r = np.polyfit(log_h, np.log(errores_r), 1)[0]

print(f"Orden de convergencia medido (Pendiente log-log):")
print(f"  - Método de Euler: {pend_e:.4f} (Teórico: 1.0)")
print(f"  - Método RK4:      {pend_r:.4f} (Teórico: 4.0)")

# Resolver con SciPy solve_ivp para control
sol_scipy_fin = solve_ivp(dv_dt, [t0, t_fin], [v0], method='RK45', rtol=1e-12, atol=1e-12)
val_scipy_fin = sol_scipy_fin.y[0][-1]
error_scipy_vs_exacto = abs(val_scipy_fin - val_exacto_fin)
print(f"\nDiferencia SciPy (RK45) vs Analítica en t={t_fin}: {error_scipy_vs_exacto:.8e}")

# Graficar
plt.figure(figsize=(8, 6))
plt.loglog(lista_h, errores_e, 'ro-', linewidth=1.5, label=f'Euler (Pendiente: {pend_e:.2f})')
plt.loglog(lista_h, errores_r, 'bs-', linewidth=1.5, label=f'RK4 (Pendiente: {pend_r:.2f})')

# Líneas de referencia
plt.loglog(lista_h, [pasos * errores_e[0]/lista_h[0] for pasos in lista_h], 'r:', alpha=0.5, label='Orden Teórico O(h)')
plt.loglog(lista_h, [(pasos**4) * errores_r[0]/(lista_h[0]**4) for pasos in lista_h], 'b:', alpha=0.5, label='Orden Teórico O(h^4)')

plt.title('Gráfico de Convergencia del Error (Escala Log-Log)', fontsize=12, fontweight='bold')
plt.xlabel('Tamaño del paso $h$')
plt.ylabel('Error absoluto en $t = 10$ s')
plt.legend(loc='upper left', frameon=True)
plt.grid(True, which="both", linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Verificaciones cuantitativas
assert np.allclose(pend_e, 1.0, atol=0.1), "La convergencia del método de Euler se desvía del valor teórico."
assert np.allclose(pend_r, 4.0, atol=0.25), "La convergencia de RK4 se desvía del valor teórico."
print("\n¡Verificación de convergencia numérica completada exitosamente!")

### Resumen de Conceptos Clave

1. **Clasificación:** Las EDOs se clasifican por su orden (mayor derivada), grado y linealidad (relación lineal de la variable dependiente y sus derivadas).
2. **Métodos Analíticos:** Las técnicas clásicas como variables separables, homogéneas, lineales, exactas, Bernoulli y Riccati permiten obtener la solución analítica exacta de la ecuación mediante integraciones algebraicas directas.
3. **Representación Geométrica:** El campo de direcciones proporciona un mapa de pendientes del plano que define cualitativamente el comportamiento del haz de soluciones sin resolver explícitamente la EDO.
4. **Esquemas Numéricos:** El método de Euler ($O(h)$) y el método RK4 ($O(h^4)$) discretizan el tiempo y aproximan las trayectorias de los PVIs.
5. **Validación de Precisión:** El análisis de convergencia y errores en escala log-log constituye una herramienta matemática fundamental para verificar el comportamiento correcto de un algoritmo numérico frente a su modelo analítico o de referencia.

### Referencias Bibliográficas

- Boyce, W. E., & DiPrima, R. C. (2012). *Elementary Differential Equations and Boundary Value Problems*. John Wiley & Sons.
- Simmons, G. F. (2016). *Differential Equations with Applications and Historical Notes*. CRC Press.
- Stewart, J. (2012). *Calculus: Early Transcendentals* (7th ed.). Cengage Learning.